# LDA Emails Analysis

**Status:** Versão de referência · **Área:** topicos-emails

Variante autogerada inicia usando pd sem importá-lo e depende de objetos não definidos no notebook.

Este arquivo está identificado como **Versão de referência**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


# Análise de Tópicos em E‑mails com LDA
Notebook gerado automaticamente a partir das instruções do professor, adaptado para o dataset `dataset_emails.csv`.

**Pipeline:** instalação de dependências → download de recursos → pré‑processamento → bigramas → LDA → avaliação.


In [ ]:
# Ajuste o caminho se necessário
data = pd.read_csv('/content/dataset_emails.csv')

# Verifique os nomes de colunas do seu CSV
print('Colunas disponíveis:', list(data.columns))

# Combine 'assunto' + 'corpo' em um único campo 'texto'
data['texto'] = data['assunto'].astype(str) + ' ' + data['corpo'].astype(str)

# Visualizar
data[['texto']].head()


In [ ]:
import re, string

stop_pt = set(stopwords.words('portuguese'))

def preprocess_text(doc, allowed_postags=None):
    # Limpar pontuação excessiva / números isolados
    doc = re.sub(r'\s+', ' ', doc)           # espaços duplos
    doc = re.sub(r'http\S+', '', doc)        # remover URLs
    doc = doc.lower()

    # Tokenização + POS com spaCy
    spacy_doc = nlp(doc)
    tokens = []
    for token in spacy_doc:
        if token.is_stop or token.is_punct or token.text in stop_pt:
            continue
        if allowed_postags and token.pos_ not in allowed_postags:
            continue
        lemma = token.lemma_.strip()
        if lemma and lemma not in string.punctuation:
            tokens.append(lemma)
    return tokens


In [ ]:
allowed_postags = ['NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV']

tqdm.pandas(desc='Pré‑processando')
data['processed'] = data['texto'].progress_apply(
    lambda x: preprocess_text(x, allowed_postags=allowed_postags)
)

data[['processed']].head()


In [ ]:
# Bigramas para captar termos compostos (e-mail, linha-de-assunto etc.)
bigram = Phrases(data['processed'], min_count=5, threshold=10)
bigram_mod = Phraser(bigram)

data['processed_bigram'] = data['processed'].apply(bigram_mod)
data[['processed_bigram']].head(3)


In [ ]:
dictionary = Dictionary(data['processed_bigram'])
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in data['processed_bigram']]

print('Número de tokens no dicionário:', len(dictionary))
print('Exemplo BoW de um documento:', corpus[0][:10])


In [ ]:
NUM_TOPICS = 8

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    chunksize=2000,
    passes=10,
    alpha='auto',
    eta='auto'
)

for idx, topic in lda_model.print_topics(-1):
    print(f'Tópico {idx}:\n{topic}\n')


In [ ]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    model=lda_model,
    texts=data['processed_bigram'],
    dictionary=dictionary,
    coherence='c_v'
)
coherence = coherence_model.get_coherence()
print(f'Coerência (c_v) para {NUM_TOPICS} tópicos: {coherence:.4f}')


In [ ]:
# Esta célula renderiza um painel interativo (pode não abrir fora do Jupyter Lab/Notebook)
pyLDAvis.enable_notebook()
viz = gensimvis.prepare(lda_model, corpus, dictionary)
viz


In [ ]:
# Descomente para salvar o modelo e o dicionário em disco
# lda_model.save('lda_emails.model')
# dictionary.save('emails_dictionary.dict')


## Continuidade

Acrescentar imports/recursos ou consolidar com versão principal; corrigir construção de Phraser e tornar independente de estado anterior.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.